In [1]:
source(here::here('MG3.functions1.R'))

# Downloading the key genomes

In [5]:
#RUN ONCE
DownloadScript="~/data/lambda/notebooks/Multigene.DownloadNCBIgenome.v2.sh"
for (gID in KeySpDF$genome){
    comToBash=paste("sbatch --gres=lscratch:30 --time 4:00:00 --mem=20g --cpus-per-task=10 -D", LogDir, DownloadScript, 
                    gID, paste0(StaticInpDir,"/keySpecies/"), sample(1:2000, 1))
    message(comToBash)
    #system(comToBash, intern = FALSE, wait = FALSE)
}

sbatch --gres=lscratch:30 --time 4:00:00 --mem=20g --cpus-per-task=10 -D ~/data/lambda/output/slurm_logs ~/data/lambda/notebooks/Multigene.DownloadNCBIgenome.v2.sh GCF_009914755.1 ~/data/lambda/data//keySpecies/ 860

sbatch --gres=lscratch:30 --time 4:00:00 --mem=20g --cpus-per-task=10 -D ~/data/lambda/output/slurm_logs ~/data/lambda/notebooks/Multigene.DownloadNCBIgenome.v2.sh GCF_000001635.27 ~/data/lambda/data//keySpecies/ 206

sbatch --gres=lscratch:30 --time 4:00:00 --mem=20g --cpus-per-task=10 -D ~/data/lambda/output/slurm_logs ~/data/lambda/notebooks/Multigene.DownloadNCBIgenome.v2.sh GCF_011100685.1 ~/data/lambda/data//keySpecies/ 552

sbatch --gres=lscratch:30 --time 4:00:00 --mem=20g --cpus-per-task=10 -D ~/data/lambda/output/slurm_logs ~/data/lambda/notebooks/Multigene.DownloadNCBIgenome.v2.sh GCF_018350175.1 ~/data/lambda/data//keySpecies/ 1126

sbatch --gres=lscratch:30 --time 4:00:00 --mem=20g --cpus-per-task=10 -D ~/data/lambda/output/slurm_logs ~/data/lambda/notebooks/M

# Temporary database for local Entrez homology table

## Homology

In [44]:
#run once
if (! file.exists(paste0(OutputDir,SQLfname))){
    comToBash=paste('sqlite3', paste0(OutputDir,EntrezSQLfname), '"VACUUM;"')
    message(comToBash)
    system(comToBash, intern = TRUE, wait = TRUE)
}
source(here::here('MG3.functions1.R')) # reloading the get the handles created


sqlite3 ~/data/lambda/output/entrez_lambda_project_reproduction.sqlite "VACUUM;"



character(0)

In [45]:
if ("entrez_homology" %in% dbListTables(EntrezdbRW)) dbExecute(EntrezdbRW, 'DROP TABLE entrez_homology')
dbExecute(EntrezdbRW, 'CREATE TABLE entrez_homology("gene_stable_id" TEXT, "protein_stable_id" TEXT, "species" TEXT, "identity" float, "homology_type" TEXT, "homology_gene_stable_id" TEXT, "homology_protein_stable_id" TEXT, "homology_species" TEXT, "homology_identity" float, "dn" TEXT, "ds" TEXT, "goc_score" float, "wga_coverage" float, "is_high_confidence" INTEGER, homology_id UNSIGNED BIG INT)')


[1] 0

In [46]:
for (f in paste0(StaticInpDir,"/keySpecies/",EntrezTSVfnames)){
    comToBash=sprintf('sqlite3 %s/%s -cmd ".mode tabs" ".import --skip 1 %s entrez_homology" ".quit"', OutputDir, EntrezSQLfname,
                     fs::path_expand(f) )
    message(comToBash)
    message(getTimeStamp())
    retval=system(comToBash, intern = TRUE, wait = TRUE)

}

sqlite3 ~/data/lambda/output//entrez_lambda_project_reproduction.sqlite -cmd ".mode tabs" ".import --skip 1 /home/frimanav/data/lambda/data/keySpecies/Compara.115.protein_default.homologies.tsv entrez_homology" ".quit"

2026-01-12_18.07.51.845745

sqlite3 ~/data/lambda/output//entrez_lambda_project_reproduction.sqlite -cmd ".mode tabs" ".import --skip 1 /home/frimanav/data/lambda/data/keySpecies/Compara.112.protein_default.homologies.tsv entrez_homology" ".quit"

2026-01-12_18.22.14.464899



In [47]:
dbExecute(EntrezdbRW, 'CREATE INDEX entrez_homologySpeciesInd ON entrez_homology(species, homology_species)')


[1] 0

In [ ]:
dbExecute(EntrezdbRW, 'CREATE INDEX entrez_homologySpeciesGeneInd ON entrez_homology(species, homology_species, gene_stable_id, homology_gene_stable_id)')


## Genes, transcripts

In [ ]:
# run once
if ("gene_transcripts" %in% dbListTables(EntrezdbRW)) dbExecute(EntrezdbRW, 'DROP TABLE gene_transcripts')

for (i in 1:nrow(KeySpDF)){
    EnsemblGTF=KeySpDF$EnsemblGTF[i]
    specie=KeySpDF$HomologueTable[i]
    if (nchar(specie)==0) specie="hsapiens"
    f = paste0(StaticInpDir,"/keySpecies/",EnsemblGTF)
    TMP.DT=getGenesTranscripsCharacteristicsFromGTFs(f)
    TMP.DT$specie=specie
    dbWriteTable(EntrezdbRW, "gene_transcripts", TMP.DT, append=TRUE)
}

In [ ]:
dbExecute(EntrezdbRW, 'CREATE INDEX gene_transcriptsSpeciesGeneInd ON gene_transcripts(specie, gene_id)')


In [ ]:
dbExecute(EntrezdbRW, 'CREATE INDEX gene_transcriptsSpeciesTranscriptInd ON gene_transcripts(specie, transcript_id)')

## Peptides

In [2]:
# run once
if ("peptides" %in% dbListTables(EntrezdbRW)) dbExecute(EntrezdbRW, 'DROP TABLE peptides')

for (i in 1:nrow(KeySpDF)){
    EnsemblFA=KeySpDF$EnsemblePepFA[i]
    specie=KeySpDF$HomologueTable[i]
    if (nchar(specie)==0) specie="hsapiens"
    f = paste0(StaticInpDir,"/keySpecies/",EnsemblFA)
    TMP.FA=readAAStringSet(f)
    PEP.SEQ=as.character(TMP.FA)
    regexmart=str_match(names(PEP.SEQ), "^([^ \\.]+)\\.([0-9]+).*gene:([^ \\.]+)\\.([0-9]+).*transcript:([^ \\.]+)\\.([0-9]+)")
    PEP.NAMES=regexmart[,2]
    PEP.VER=regexmart[,3]
    GENE.NAMES=regexmart[,4]
    GENE.VER=regexmart[,5]
    TRN.NAMES=regexmart[,6]
    TRN.VER=regexmart[,7]
    names(PEP.SEQ)=NULL
    TMP.DF=data.frame(peptide_id=PEP.NAMES, peptide_ver=as.numeric(PEP.VER), transcript_id=TRN.NAMES, transcript_ver=as.numeric(TRN.VER),
                      gene_id=GENE.NAMES, gene_ver=GENE.VER,
                      peptide=PEP.SEQ)
    TMP.DF$specie=specie
    dbWriteTable(EntrezdbRW, "peptides", TMP.DF, append=TRUE)
}

In [3]:
dbExecute(EntrezdbRW, 'CREATE INDEX peptidesSpeciesTranscriptInd ON peptides(specie, transcript_id)')

[1] 0

In [4]:
dbExecute(EntrezdbRW, 'CREATE INDEX peptidesSpeciesGeneInd ON peptides(specie, peptide_id)')

[1] 0

## Gene synonims

### Downloading the files

In [51]:
library(rvest)

In [115]:
table_schemas=list()
table_schemas[["gene"]]=list()
table_schemas[["gene"]][["cols"]]=c("gene_id", "biotype", "analysis_id", "seq_region_id", "seq_region_start", 
                                 "seq_region_end", "seq_region_strand", "display_xref_id", "source", "description", 
                                 "is_current", "canonical_transcript_id", "stable_id", "version", "created_date",
                                 "modified_date")
table_schemas[["gene"]][["types"]]=c('UNSIGNED BIG INT', 'TEXT', 'TEXT', 'TEXT', 'TEXT','TEXT', 'TEXT', 'UNSIGNED BIG INT', 'TEXT', 'TEXT', 'TEXT', 'TEXT', 'TEXT', 'TEXT', 'TEXT', 'TEXT')
table_schemas[["gene"]][["index"]]=c('gene_id', 'stable_id', 'display_xref_id')


table_schemas[["gene_attrib"]]=list()
table_schemas[["gene_attrib"]][["cols"]]=c("gene_id", "attrib_type_id", "value")
table_schemas[["gene_attrib"]][["types"]]=c("UNSIGNED BIG INT", "UNSIGNED BIG INT", "TEXT")
table_schemas[["gene_attrib"]][["index"]]=c('gene_id', 'attrib_type_id', 'value')


table_schemas[["attrib_type"]]=list()
table_schemas[["attrib_type"]][["cols"]]=c("attrib_type_id", "code", "name", "description")
table_schemas[["attrib_type"]][["types"]]=c("UNSIGNED BIG INT", "TEXT", "TEXT", "TEXT")
table_schemas[["attrib_type"]][["index"]]=c('attrib_type_id', 'name')

table_schemas[["external_synonym"]]=list()
table_schemas[["external_synonym"]][["cols"]]=c('xref_id', 'synonym')
table_schemas[["external_synonym"]][["types"]]=c('UNSIGNED BIG INT', 'TEXT')
table_schemas[["external_synonym"]][["index"]]=c('xref_id', 'synonym')


table_schemas[["xref"]]=list()
table_schemas[["xref"]][["cols"]]=c("xref_id", "external_db_id", "dbprimary_acc", "display_label", "version", "description", "info_type", "info_text")
table_schemas[["xref"]][["types"]]=c('UNSIGNED BIG INT', "TEXT", "TEXT", "TEXT", "TEXT", "TEXT", "TEXT", "TEXT")
table_schemas[["xref"]][["index"]]=c('xref_id', 'display_label')



In [118]:
for (i in 1:nrow(KeySpDF)){
    message(KeySpDF$ScientificName[i])
    ftppath=KeySpDF$EnsembleFTPpath[i]
    speciedir=get_table_prefix(ftppath)
    tmpdir=paste0(StaticInpDir,"keySpecies/",speciedir,"/")
    system(paste0("mkdir ", tmpdir, " -p"), intern = TRUE, wait = TRUE)
    file_list=paste0(names(table_schemas),".txt.gz")
    for (f in file_list){
        message("\t",f)
        destfile=paste0(tmpdir, f)
        unarch_destfile=gsub(pattern = ".gz", replacement = "", x = destfile, fixed = TRUE)
        if ((! file.exists(destfile)) & (! file.exists(unarch_destfile))){
            download.file(url = paste0(ftppath, f), destfile = destfile)
            Sys.sleep(2) 
        }
    }
    system(paste0("gunzip ", tmpdir, "/*.gz"), intern = TRUE, wait = TRUE)
    # saving to SQL
    message("\tSaving to SQL")
    for (f in file_list){
        
        unarch_fname=gsub(pattern = ".gz", replacement = "", x = f, fixed = TRUE)
        source_tab_file=paste0(tmpdir, unarch_fname)
        initial_table_name=gsub(pattern = ".txt", replacement = "", x = unarch_fname, fixed = TRUE)
        tablename=paste0(speciedir,"_",initial_table_name)
        message('\t',initial_table_name)
        queryTxt=sprintf('CREATE TABLE %s(', tablename)
        for (j in 1:length(table_schemas[[initial_table_name]][['cols']])){
            column_name=table_schemas[[initial_table_name]][['cols']][j]
            column_type=table_schemas[[initial_table_name]][['types']][j]
            queryTxt=sprintf('%s "%s" %s,',queryTxt, column_name, column_type)
        }
        
        queryTxt=substr(x = queryTxt, start = 1, stop = nchar(queryTxt)-1)
        queryTxt=paste0(queryTxt,')')
        message('\t',queryTxt)
        dbExecute(EntrezdbRW, queryTxt)
        # import 
        comToBash=sprintf('sqlite3 %s/%s -cmd ".mode tabs" ".import %s %s" ".quit"', OutputDir, EntrezSQLfname,
                     fs::path_expand(source_tab_file), tablename )
        message('\t',comToBash)
        message('\t',getTimeStamp())
        retval=system(comToBash, intern = TRUE, wait = TRUE)
        # creating indexes
        for (k in 1:length(table_schemas[[initial_table_name]][['index']])){
            indexName=paste0(tablename,"Ind",k)
            indexvalues=table_schemas[[initial_table_name]][['index']][k]
            queryTxt=sprintf('CREATE INDEX %s ON %s(%s)', indexName, tablename, indexvalues)
            message('\t',queryTxt)
            message(getTimeStamp())
            dbExecute(EntrezdbRW, queryTxt)
        }
    }
}

homo_sapiens

	gene.txt.gz

	gene_attrib.txt.gz

	attrib_type.txt.gz

	external_synonym.txt.gz

	xref.txt.gz

Warning message in system(paste0("gunzip ", tmpdir, "/*.gz"), intern = TRUE, wait = TRUE):
“running command 'gunzip ~/data/lambda/data/keySpecies/homo_sapiens_core_115_38//*.gz' had status 1”
	Saving to SQL

	gene

	CREATE TABLE homo_sapiens_core_115_38_gene( "gene_id" UNSIGNED BIG INT, "biotype" TEXT, "analysis_id" TEXT, "seq_region_id" TEXT, "seq_region_start" TEXT, "seq_region_end" TEXT, "seq_region_strand" TEXT, "display_xref_id" UNSIGNED BIG INT, "source" TEXT, "description" TEXT, "is_current" TEXT, "canonical_transcript_id" TEXT, "stable_id" TEXT, "version" TEXT, "created_date" TEXT, "modified_date" TEXT)

	sqlite3 ~/data/lambda/output//entrez_lambda_project_reproduction.sqlite -cmd ".mode tabs" ".import /home/frimanav/data/lambda/data/keySpecies/homo_sapiens_core_115_38/gene.txt homo_sapiens_core_115_38_gene" ".quit"

	2026-01-13_16.11.46.147841

	CREATE INDEX homo_sapie

[1] "CREAT"

In [116]:
comToBash=sprintf('sqlite3 %s/%s -cmd ".mode tabs" ".import --skip 1 %s %s" ".quit"', OutputDir, EntrezSQLfname,
                     fs::path_expand(source_tab_file), tablename )
        message(comToBash)

sqlite3 ~/data/lambda/output//entrez_lambda_project_reproduction.sqlite -cmd ".mode tabs" ".import --skip 1 /home/frimanav/data/lambda/data/keySpecies/homo_sapiens_core_115_38/gene.txt homo_sapiens_core_115_38_gene" ".quit"



[1] "balaenoptera_musculus_core_115_2_xref"

# Creating the empty database

In [5]:
#run once
if (! file.exists(paste0(OutputDir,SQLfname))){
    comToBash=paste('sqlite3', paste0(OutputDir,SQLfname), '"VACUUM;"')
    message(comToBash)
    system(comToBash, intern = TRUE, wait = TRUE)
}
source(here::here('MG3.functions1.R')) # reloading the get the handles created


sqlite3 ~/data/lambda/output/lambda_project_reproduction.sqlite "VACUUM;"



character(0)

## Filling up the database with homology tables from Entrez

In [10]:
# run ONCE

if ("homology" %in% dbListTables(mydbRW)) dbExecute(mydbRW, 'DROP TABLE homology')

for (i in 2:nrow(KeySpDF)){
    martid = KeySpDF$HomologueTable[i]
    scientific_name= KeySpDF$ScientificName[i]
    s_name=martid
    if (nchar(s_name)==0) s_name="hsapiens"
    message(martid)
    queryTxt=sprintf('SELECT DISTINCT gene_stable_id AS ensembl_gene_id, homology_gene_stable_id AS homolog FROM entrez_homology WHERE species="homo_sapiens" AND homology_species="%s" AND homology_gene_stable_id IN (SELECT gene_id FROM gene_transcripts WHERE specie="%s" AND gene_biotype="protein_coding") AND ( homology_identity>55 OR is_high_confidence=1)', scientific_name, s_name)
    tmpDF=dbGetQuery(EntrezdbRO, queryTxt)
    tmpDF=tmpDF[tmpDF$homolog!="",]
    tmpDF=tmpDF[!is.na(tmpDF$homolog),]
    tmpDF$specie=martid
    queryTxt=sprintf('SELECT DISTINCT gene_stable_id AS homolog, homology_gene_stable_id AS ensembl_gene_id FROM entrez_homology WHERE homology_species="homo_sapiens" AND species="%s" AND gene_stable_id IN (SELECT gene_id FROM gene_transcripts WHERE specie="%s" AND gene_biotype="protein_coding") AND ( homology_identity>55 OR is_high_confidence=1)', scientific_name, s_name)
    tmpDF2=dbGetQuery(EntrezdbRO, queryTxt)
    if (nrow(tmpDF2)>0){
        tmpDF2=tmpDF2[tmpDF2$homolog!="",]
        tmpDF2=tmpDF2[!is.na(tmpDF2$homolog),]
        tmpDF2$specie=martid
        tmpDF3= rbind(tmpDF, tmpDF2)
    }else{
        tmpDF3=tmpDF
    }
    tmpDF3= tmpDF3 %>% distinct()
    dbWriteTable(mydbRW, "homology", tmpDF3, append=TRUE)

}
dbExecute(mydbRW, 'CREATE INDEX homologyHumanGeneInd ON homology(ensembl_gene_id)')
dbExecute(mydbRW, 'CREATE INDEX homologySpecieInd ON homology(specie, ensembl_gene_id)')
dbExecute(mydbRW, 'CREATE INDEX homologyHomologInd ON homology(homolog, specie)')


[1] 0

mmusculus

clfamiliaris

fcatus

btaurus

cdromedarius

lafricana

sscrofa

bmusculus



[1] 0

[1] 0

[1] 0

In [2]:
# testing if bijective homology view works
source(here::here('MG3.functions1.R')) # reloading the get the views generated
queryTxt='SELECT * FROM BijectiveRelevantSpeciesView' 
BijectiveHomology=dbGetQuery(mydbRO, queryTxt)
message(nrow(BijectiveHomology))
message(length(unique(BijectiveHomology$ensembl_gene_id)))

78776

9847



# Building HMM models

## Downloading amino acid sequences into the database

In [3]:
#run ONCE
if ("keyAAsequences" %in% dbListTables(mydbRW)) dbExecute(mydbRW, 'DROP TABLE keyAAsequences')

for (sp in KeySpDF$HomologueTable){
    if (nchar(sp)==0){
        spname="hsapiens"
        queryTxt='SELECT DISTINCT ensembl_gene_id FROM BijectiveRelevantSpeciesView'
        geneListTMP=dbGetQuery(mydbRO, queryTxt)[[1]]
    }else{
        spname=sp
        queryTxt=sprintf('SELECT DISTINCT homolog FROM BijectiveRelevantSpeciesView WHERE specie="%s"', spname)
        geneListTMP=dbGetQuery(mydbRO, queryTxt)[[1]]
    }
    message(spname)
    if ("keyAAsequences" %in% dbListTables(mydbRW)){
        queryTxt=sprintf('SELECT COUNT(ensembl_gene_id) FROM keyAAsequences WHERE specie="%s"', spname) 
        retval=dbGetQuery(mydbRW, queryTxt)
        if (retval[1,1]>0){
            message("Found in DB")
            next
        }
    }
    message("Requesting the data")
    tmp_aaDT=retrieveSpecieProteinsLocalSQL(spname, geneListTMP)
    dbWriteTable(mydbRW, "keyAAsequences", tmp_aaDT, append=TRUE)
}

dbExecute(mydbRW, 'CREATE INDEX keyAAsequencesGeneInd ON keyAAsequences(ensembl_gene_id)')
dbExecute(mydbRW, 'CREATE INDEX keyAAsequencesSpecieInd ON keyAAsequences(specie, ensembl_gene_id)')
dbExecute(mydbRW, 'CREATE INDEX keyAAsequencesTranscrInd ON keyAAsequences(ensembl_transcript_id)')
dbExecute(mydbRW, 'CREATE INDEX keyAAsequencesPeptInd ON keyAAsequences(ensembl_peptide_id)')

hsapiens

Requesting the data

hsapiens

Peptides query

Length filter

Key 2

Key 3

Merge in function

mmusculus

Requesting the data

mmusculus

Peptides query

Length filter

Key 2

Key 3

Merge in function

clfamiliaris

Requesting the data

clfamiliaris

Peptides query

Length filter

Key 2

Key 3

Merge in function

fcatus

Requesting the data

fcatus

Peptides query

Length filter

Key 2

Key 3

Merge in function

btaurus

Requesting the data

btaurus

Peptides query

Length filter

Key 2

Key 3

Merge in function

cdromedarius

Requesting the data

cdromedarius

Peptides query

Length filter

Key 2

Key 3

Merge in function

lafricana

Requesting the data

lafricana

Peptides query

Length filter

Key 2

Key 3

Merge in function

sscrofa

Requesting the data

sscrofa

Peptides query

Length filter

Key 2

Key 3

Merge in function

bmusculus

Requesting the data

bmusculus

Peptides query

Length filter

Key 2

Key 3

Merge in function



[1] 0

[1] 0

[1] 0

[1] 0

In [4]:
# tested singular definitions by
dbGetQuery(mydbRO, 'SELECT ensembl_gene_id, numpept FROM ( SELECT ensembl_gene_id, COUNT( peptide ) AS numpept FROM keyAAsequences GROUP BY ensembl_gene_id ) WHERE numpept > 1')
# no rows returned

ensembl_gene_id,numpept
<chr>,<lgl>


## Getting AA sequences groupped

In [5]:
queryTxt="SELECT ensembl_gene_id, peptide AS hsapiensPeptide
    FROM keyAAsequences WHERE ensembl_gene_id IN (SELECT DISTINCT ensembl_gene_id FROM BijectiveRelevantSpeciesView ) AND
    specie='hsapiens'"
totalAAseqDT=dbGetQuery(mydbRO, queryTxt)

for (sp in relevantSpecies){
    message(sp)
    queryTxt=sprintf("SELECT keyAAsequences.ensembl_gene_id AS %sHomolog, BijectiveRelevantSpeciesView.ensembl_gene_id AS ensembl_gene_id, 
    peptide AS %sPeptide
    FROM keyAAsequences INNER JOIN BijectiveRelevantSpeciesView ON BijectiveRelevantSpeciesView.homolog = keyAAsequences.ensembl_gene_id 
    WHERE keyAAsequences.specie='%s' AND peptide_length > 0",sp, sp,sp)
    TMPdt=dbGetQuery(mydbRO, queryTxt)
    totalAAseqDT=merge(totalAAseqDT, TMPdt, by="ensembl_gene_id", all=FALSE)
    message("\t Number of peptides to export nrow(totalAAseqDT): ", nrow(totalAAseqDT))
}

mmusculus

	 Number of peptides to export nrow(totalAAseqDT): 9847

clfamiliaris

	 Number of peptides to export nrow(totalAAseqDT): 9847

fcatus

	 Number of peptides to export nrow(totalAAseqDT): 9847

btaurus

	 Number of peptides to export nrow(totalAAseqDT): 9847

cdromedarius

	 Number of peptides to export nrow(totalAAseqDT): 9847

lafricana

	 Number of peptides to export nrow(totalAAseqDT): 9847

sscrofa

	 Number of peptides to export nrow(totalAAseqDT): 9847

bmusculus

	 Number of peptides to export nrow(totalAAseqDT): 9847



In [6]:
GenerateFastaRecord=function(inrow){
    SpecieNames=names(inrow)[ grepl("peptide", names(inrow), ignore.case = TRUE)]
    SpecieNames=str_replace(string = SpecieNames, pattern = "Peptide", replacement = "")
    outRecord=""
    for ( sp in SpecieNames){
        AAseq=inrow[[paste0(sp, "Peptide")]]
        AAseq=str_remove_all(string = AAseq, pattern = "\\*")
        if (sp == 'hsapiens'){
            GeneID=inrow$ensembl_gene_id
        }else{
            GeneID=inrow[[paste0(sp,"Homolog")]]
        }
        tmpRecord=paste0(">",sp," ",GeneID ,"\n", AAseq, "\n")
        outRecord=paste0(outRecord, tmpRecord)
    }
    return(outRecord)
}

In [7]:
totalAAseqDT = totalAAseqDT %>% rowwise() %>% mutate(FArecord=GenerateFastaRecord(cur_data()))

Warning message:
“There was 1 warning in `mutate()`.
ℹ In argument: `FArecord = GenerateFastaRecord(cur_data())`.
ℹ In row 1.
Caused by warning:
! `cur_data()` was deprecated in dplyr 1.1.0.
ℹ Please use `pick()` instead.”


In [9]:
dir.create(paste0(OutputDir,"HMMs/"), showWarnings = FALSE)

for (i in 1:nrow(totalAAseqDT)){
    if (i %% 1000 == 0) message(i,"/", nrow(totalAAseqDT))
    fname=paste0(OutputDir,"HMMs/",totalAAseqDT$ensembl_gene_id[i],".faa")
    FAtext=totalAAseqDT$FArecord[i]
    writeLines(FAtext, fname, sep="")
}

1000/9847

2000/9847

3000/9847

4000/9847

5000/9847

6000/9847

7000/9847

8000/9847

9000/9847



## Running MUSCLE

In [10]:
dir.create(paste0(OutputDir,"HMMs/slurm_logs/"), showWarnings = FALSE)
sbatchscript="~/data/lambda/notebooks/MG3.Step1.MuscleAndHMMER.sh"
fs=list.files(paste0(OutputDir,"HMMs/"), pattern = "ENSG.*faa$", full.names = TRUE)
swarmlines=NULL 
for (fname in fs){
    swarmline=paste0(sbatchscript," ", fname)
    swarmlines=addToVec(swarmlines, swarmline)

}

maxchunk=80
timestamp=getTimeStamp()
processed=0
i=0
shFiles=NULL
while (processed<length(swarmlines)){
    upperlimit=processed+min(maxchunk, length(swarmlines)-processed)
    #message(paste(processed+1, upperlimit))
    fname=paste0(OutputDir,"HMMs/runningMuscleAndHMMER.",i,"-", timestamp,".sh")
    writeLines(text = swarmlines[(processed+1):upperlimit], 
               con=fname)
    shFiles=addToVec(shFiles, fname)
    processed=upperlimit
    i=i+1
}

system(paste0("chmod a+x ",OutputDir,"HMMs/runningMuscleAndHMMER.*.sh"), intern = FALSE, wait = TRUE)

swarmparams=paste0("#SWARM -t 6 -g 40 --time=00:40:00 --gres=lscratch:20 --logdir ", fs::path_expand(OutputDir),"/HMMs/slurm_logs/")
timestamp=getTimeStamp()
swarmfname=paste0(OutputDir,"HMMs/runningMuscleAndHMMER-", timestamp,".swarm")
writeLines(text = c(swarmparams, shFiles),  con=swarmfname)

message("Run from bash: ", swarmfname)

Run from bash: ~/data/lambda/output/HMMs/runningMuscleAndHMMER-2026-01-21_12.00.45.730353.swarm



## Checking that the HMM models point only to itself and do not produce false positive hits to other proteins

### Exporting key AA sequences from SQL to FastA

In [18]:
saveKeySpFAArecords=function(sp, FAArecords){
    tmpDir=paste0(OutputDir,"HMMsKeySpeciesResults/",sp,"/")
    system(paste0("mkdir -p ", tmpDir), wait=TRUE)
    writeLines(FArecords, con = paste0(tmpDir, "protein.faa"))
}

system(paste0("mkdir -p ", paste0(OutputDir,"HMMsKeySpeciesResults/slurm_logs/")), wait=TRUE)

for (sp in relevantSpecies){
    queryTxt=sprintf('SELECT BijectiveRelevantSpeciesView.ensembl_gene_id AS ensembl_gene_id,
                     keyAAsequences.ensembl_gene_id AS specieGeneID, peptide FROM keyAAsequences 
                     INNER JOIN BijectiveRelevantSpeciesView ON 
                     keyAAsequences.ensembl_gene_id=BijectiveRelevantSpeciesView.homolog 
                     WHERE keyAAsequences.specie="%s"',
                     sp) 
    specieAAdf=dbGetQuery(mydbRO, queryTxt)
    AAseq=specieAAdf$peptide
    AAseq=str_remove_all(string = AAseq, pattern = "\\*")
    FArecords=paste0(">",specieAAdf$ensembl_gene_id, " ", specieAAdf$specieGeneID,"\n", AAseq)
    saveKeySpFAArecords(sp, FArecords)
}

queryTxt='SELECT ensembl_gene_id, peptide FROM keyAAsequences 
                     WHERE ensembl_gene_id IN ( SELECT DISTINCT ensembl_gene_id FROM BijectiveRelevantSpeciesView) 
                    AND keyAAsequences.specie="hsapiens"'
specieAAdf=dbGetQuery(mydbRO, queryTxt)
AAseq=specieAAdf$peptide
AAseq=str_remove_all(string = AAseq, pattern = "\\*")
FArecords=paste0(">",specieAAdf$ensembl_gene_id, "\n", AAseq)
saveKeySpFAArecords("hsapiens", FArecords)

### Running HMMsearch over the key species

In [19]:
# creating chunks of the HMM files
sbatchscript="~/data/lambda/notebooks/runHMMsearchSpeciev2.sh"

max_chunk=400
fs=list.files(paste0(OutputDir,"HMMs/"), pattern = "ENSG.*MSA.faa.hmm$", full.names = TRUE)
startpos=1
i=0
swarmlines=NULL

while (startpos<length(fs)){
    endpos=min(startpos+max_chunk,length(fs))
    fname=paste0(OutputDir,"/HMMsKeySpeciesResults/filelist-",i,".txt")
    writeLines(text = fs[startpos:endpos], 
               con=fname)

    for (sp in c("hsapiens", relevantSpecies)){
        swarmline=paste0(sbatchscript," ", sp, " ", OutputDir,"/HMMsKeySpeciesResults/ ", 
                         fname," ", maxPvalue, " ", OutputDir,"/HMMsKeySpeciesResults/",sp,"/")
        swarmlines=addToVec(swarmlines, swarmline)
    }
    
    startpos=endpos+1
    i=i+1
}

In [20]:
swarmline

[1] "~/data/lambda/notebooks/runHMMsearchSpeciev2.sh bmusculus ~/data/lambda/output//HMMsKeySpeciesResults/ ~/data/lambda/output//HMMsKeySpeciesResults/filelist-24.txt 1E-10 ~/data/lambda/output//HMMsKeySpeciesResults/bmusculus/"

In [21]:
swarmparams=paste0("#SWARM -t 5 -g 10 --time=1:00:00 --gres=lscratch:10  --logdir ", path.expand(OutputDir),"HMMsKeySpeciesResults/slurm_logs/")
maxchunk=420
timestamp=getTimeStamp()
processed=0
i=0
swarmFiles=NULL
while (processed<length(swarmlines)){
    upperlimit=processed+min(maxchunk, length(swarmlines)-processed)
    message(paste(processed+1, upperlimit))
    fname=paste0(OutputDir,"HMMsKeySpeciesResults/runningHMMsearch.",i,"-", timestamp,".swarm")
    writeLines(text = c(swarmparams, swarmlines[(processed+1):upperlimit]), 
               con=fname)
    message("Run from bash: ", fname)
    swarmFiles=addToVec(swarmFiles, fname)
    processed=upperlimit
    i=i+1
}

1 225

Run from bash: ~/data/lambda/output/HMMsKeySpeciesResults/runningHMMsearch.0-2026-01-21_13.22.08.516995.swarm



In [22]:
# run the HMMsKeySpeciesResults/runningHMMsearch.*.swarm file in bash

### Analyzing the HMM search results

In [23]:
# run ONCE

if ("HMMkeySpecieCrossHits" %in% dbListTables(mydbRW)) dbExecute(mydbRW, 'DROP TABLE HMMkeySpecieCrossHits')

for (sp in c("hsapiens", relevantSpecies)){
    message(sp)
    SpecieOutDir=paste0(OutputDir, "HMMsKeySpeciesResults/", sp,"/")
    HMMouts=list.files(path = SpecieOutDir, pattern = paste0("*.hmm.out"), full.names = TRUE)
    specieDTtmp=NULL
    for ( i in 1:length(HMMouts)){
        if (i %% 1000 == 0){
            message(paste("\t", i,"/",length(HMMouts)))
        }
        HMMfname=HMMouts[i]
        #message(HMMfname)
        HMM.DT=readHMMout(HMMfname)
        if (!is.null(HMM.DT)){
            HMM.DT=HMM.DT[ HMM.DT$score >= max(HMM.DT$score)*minHMMscoreRatioKeySpecies, ]
            specieDTtmp=addToDF(specieDTtmp, HMM.DT)
        }
    }
    specieDTtmp=renameDFcolumn(specieDTtmp, "targetname", "hit")
    specieDTtmp=renameDFcolumn(specieDTtmp, "queryname", "HMM")
    specieDTtmp$specie=sp
    specieDTtmp=specieDTtmp[, c("HMM", "hit", "specie", "Evalue", "score", "bias")]
    specieDTtmp$HMM=str_remove_all(specieDTtmp$HMM, pattern = "\\.faa\\.MSA")
    specieDTtmp=specieDTtmp[ specieDTtmp$HMM!=specieDTtmp$hit ,]
    dbWriteTable(mydbRW, "HMMkeySpecieCrossHits", specieDTtmp, append=TRUE)
}

dbExecute(mydbRW, 'CREATE INDEX HMMkeySpecieCrossHitsHMMInd ON HMMkeySpecieCrossHits(HMM, specie)')
dbExecute(mydbRW, 'CREATE INDEX HMMkeySpecieCrossHitsHitInd ON HMMkeySpecieCrossHits(hit, specie)')
dbExecute(mydbRW, 'CREATE INDEX HMMkeySpecieCrossHitsHitHMMInd ON HMMkeySpecieCrossHits(hit, HMM, specie)')
dbExecute(mydbRW, 'CREATE INDEX HMMkeySpecieCrossHitsHitSpecieInd ON HMMkeySpecieCrossHits(specie, HMM, hit)')

hsapiens

	 1000 / 9847

	 2000 / 9847

	 3000 / 9847

	 4000 / 9847

	 5000 / 9847

	 6000 / 9847

	 7000 / 9847

	 8000 / 9847

	 9000 / 9847

mmusculus

	 1000 / 9847

	 2000 / 9847

	 3000 / 9847

	 4000 / 9847

	 5000 / 9847

	 6000 / 9847

	 7000 / 9847

	 8000 / 9847

	 9000 / 9847

clfamiliaris

	 1000 / 9847

	 2000 / 9847

	 3000 / 9847

	 4000 / 9847

	 5000 / 9847

	 6000 / 9847

	 7000 / 9847

	 8000 / 9847

	 9000 / 9847

fcatus

	 1000 / 9847

	 2000 / 9847

	 3000 / 9847

	 4000 / 9847

	 5000 / 9847

	 6000 / 9847

	 7000 / 9847

	 8000 / 9847

	 9000 / 9847

btaurus

	 1000 / 9847

	 2000 / 9847

	 3000 / 9847

	 4000 / 9847

	 5000 / 9847

	 6000 / 9847

	 7000 / 9847

	 8000 / 9847

	 9000 / 9847

cdromedarius

	 1000 / 9847

	 2000 / 9847

	 3000 / 9847

	 4000 / 9847

	 5000 / 9847

	 6000 / 9847

	 7000 / 9847

	 8000 / 9847

	 9000 / 9847

lafricana

	 1000 / 9847

	 2000 / 9847

	 3000 / 9847

	 4000 / 9847

	 5000 / 9847

	 6000 / 9847

	 7000 / 9847

	 8000 /

[1] 0

[1] 0

[1] 0

[1] 0

In [24]:
source(here::here('MG3.functions1.R')) # reloading the get the views generated
tmpDF2=dbGetQuery(mydbRO, "SELECT * FROM HomologyBijectiveWellDefined")
message("Number of well defined proteins: ", length(unique(tmpDF2$ensembl_gene_id)))

Number of well defined proteins: 9754



# Building gene name synonim table

In [183]:
getEnsemblSynonymsLocalSQL=function(specieInd, inValues){
    outDF=NULL
    # 1 gene attrib
    ftppath=KeySpDF$EnsembleFTPpath[specieInd]
    table_prefix=get_table_prefix(ftppath)
    inSpecie=KeySpDF$HomologueTable[specieInd]
    if (nchar(inSpecie)==0) inSpecie="hsapiens"
    queryTxt=sprintf("SELECT genetbl.stable_id AS specieEnsemblID, atrtbl.value AS synonym FROM 
                     %s_gene_attrib AS atrtbl INNER JOIN %s_gene AS genetbl 
                     ON atrtbl.gene_id = genetbl.gene_id  WHERE attrib_type_id IN 
                     (SELECT attrib_type_id FROM %s_attrib_type WHERE name = 'name' OR name='Name')
                     AND genetbl.biotype='protein_coding' AND
                     genetbl.stable_id IN (SELECT gene_id FROM 
                         gene_transcripts WHERE specie='%s' AND gene_biotype='protein_coding');",table_prefix,table_prefix,table_prefix, inSpecie)
    tmpDF1=dbGetQuery(EntrezdbRO, queryTxt)
    
    if (nrow(tmpDF1)>0){
        tmpDF1$specie=inSpecie
        tmpDF1$column='gene_attrib'
        outDF=tmpDF1
    }
    # 2 external_synonym
    queryTxt=sprintf("SELECT genetbl.stable_id AS specieEnsemblID, extsymtbl.synonym AS synonym FROM 
                     %s_external_synonym AS extsymtbl INNER JOIN %s_gene AS genetbl 
                     ON extsymtbl.xref_id = genetbl.display_xref_id WHERE 
                    genetbl.biotype='protein_coding' AND
                     genetbl.stable_id IN (SELECT gene_id FROM 
                         gene_transcripts WHERE specie='%s' AND gene_biotype='protein_coding');",table_prefix,table_prefix, inSpecie)
    tmpDF2=dbGetQuery(EntrezdbRO, queryTxt)
    if (nrow(tmpDF2)>0){
        tmpDF2$specie=inSpecie
        tmpDF2$column='external_synonym'
        outDF=addToDF(outDF, tmpDF2)
    }
    # 2 xref display_label
    queryTxt=sprintf("SELECT genetbl.stable_id AS specieEnsemblID, xref.display_label AS synonym FROM 
                     %s_xref AS xref INNER JOIN %s_gene AS genetbl 
                     ON xref.xref_id = genetbl.display_xref_id WHERE 
                    genetbl.biotype='protein_coding' AND
                     genetbl.stable_id IN (SELECT gene_id FROM 
                         gene_transcripts WHERE specie='%s' AND gene_biotype='protein_coding');",table_prefix,table_prefix, inSpecie)
    tmpDF3=dbGetQuery(EntrezdbRO, queryTxt)
    if (nrow(tmpDF3)>0){
        tmpDF3$specie=inSpecie
        tmpDF3$column='xref.display_label'
        outDF=addToDF(outDF, tmpDF3)
    }
    # filtering synonims which are ambigous
    outDF = outDF %>% group_by(synonym) %>% filter(length(unique(specieEnsemblID))==1) %>% ungroup()
    # adding human homolog
    #if (inSpecie=="hsapiens"){
    #    outDF$ensembl_gene_id=outDF$specieEnsemblID
    #}else{
    #    # getting the bijective DF
    #    queryTxt=sprintf('SELECT * FROM BijectiveRelevantSpeciesView WHERE specie="%s"', inSpecie)
    #    homologyDF=dbGetQuery(mydbRO, queryTxt)
    #    homologyDF$specie=NULL
    #    outDF=merge(outDF, homologyDF, by.x = "specieEnsemblID", by.y = "homolog", all = FALSE)
    #}
    outDF=outDF[ outDF$specieEnsemblID %in% inValues, ]
    return(outDF)
}

In [185]:
# human  treated seprately
#run ONCE
if ("GeneNameSynonims" %in% dbListTables(mydbRW)) dbExecute(mydbRW, 'DROP TABLE GeneNameSynonims')

AllSpeciesSynonimsDF <- data.frame(
  specie = character(0),
  ensembl_gene_id = character(0),
  specieEnsemblID = character(0),
  synonym = character(0)
)


for (specieind in 1:nrow(KeySpDF)){
    sp=KeySpDF$HomologueTable[specieind]
    if (nchar(sp)==0){
        sp="hsapiens"
        message(sp)
        queryTxt='SELECT DISTINCT ensembl_gene_id FROM homology'
        ENSidsDF=dbGetQuery(mydbRO, queryTxt)
        specieSynonymsTMP=getEnsemblSynonymsLocalSQL(specieind, ENSidsDF$ensembl_gene_id)
        AllSpeciesSynonimsDF=specieSynonymsTMP
        AllSpeciesSynonimsDF$ensembl_gene_id = AllSpeciesSynonimsDF$specieEnsemblID
    }else{
        message(sp)
        if (sp %in% AllSpeciesSynonimsDF$specie){
            message("Found. next")
            next
        }
        queryTxt=sprintf('SELECT ensembl_gene_id, homolog AS specieEnsemblID FROM homology WHERE specie="%s"', sp)
        ENSidsDF=unique(dbGetQuery(mydbRO, queryTxt))
        specieSynonymsTMP=getEnsemblSynonymsLocalSQL(specieind, ENSidsDF$specieEnsemblID)
        specieSynonymsTMP = merge( ENSidsDF, specieSynonymsTMP, by="specieEnsemblID", all=FALSE)
        specieSynonymsTMP = specieSynonymsTMP %>% group_by(synonym) %>% filter(length(unique(ensembl_gene_id))==1) %>% ungroup()
        AllSpeciesSynonimsDF=addToDF(AllSpeciesSynonimsDF,specieSynonymsTMP)
    }
}
AllSpeciesSynonimsDF = AllSpeciesSynonimsDF %>% group_by(synonym) %>% filter(length(unique(ensembl_gene_id))==1) %>% ungroup()

dbWriteTable(mydbRW, "GeneNameSynonims", AllSpeciesSynonimsDF, append=FALSE)
dbExecute(mydbRW, 'CREATE INDEX GeneNameSynonimsspecieEnsemblIDInd ON GeneNameSynonims(specieEnsemblID)')
dbExecute(mydbRW, 'CREATE INDEX GeneNameSynonimssynonymInd ON GeneNameSynonims(synonym)')
dbExecute(mydbRW, 'CREATE INDEX GeneNameSynonimsspecieInd ON GeneNameSynonims(specie, specieEnsemblID)')
dbExecute(mydbRW, 'CREATE INDEX GeneNameSynonimsensembl_gene_idInd ON GeneNameSynonims(ensembl_gene_id)')

hsapiens

mmusculus

clfamiliaris

fcatus

btaurus

cdromedarius

lafricana

sscrofa

bmusculus



[1] 0

[1] 0

[1] 0

[1] 0

# Download the genomes

## Get list of the genomes

In [11]:
comToBash="~/scripts/datasets --api-key " 
comToBash=paste0(comToBash, apikey, " summary genome taxon eutheria --assembly-level chromosome,complete --annotated --exclude-atypical ")
comToBash=paste0(comToBash, " > ",StaticInpDir,"/","assemblies.json")
system(comToBash, intern = TRUE, wait = TRUE)

character(0)

## Generate a swarm file

In [22]:
assembyfilename=paste0(StaticInpDir,"assemblies.json")
assembltext=readLines(assembyfilename)
assembliesJSON <- fromJSON(assembltext)

In [37]:
swarmparams="#SWARM -t 5 -g 10 --time=10:00:00 --gres=lscratch:10"
scriptname="~/data/lambda/notebooks/download_genome.sh"
swarmlines=c(swarmparams)
for (i in 1:length(assembliesJSON$reports)){
    assemblyID=assembliesJSON$reports[[i]]$accession
    tmpline=paste0(scriptname," ", assemblyID, " ", StaticInpDir, " ", sample(1:1000, 1))
    swarmlines=addToVec(swarmlines, tmpline)
}

fname=paste0(StaticInpDir,"download_genomes-", getTimeStamp(),".swarm")
writeLines(text = swarmlines, con=fname)
message("resulting file: ", fname, " \n run it from bash")

resulting file: ~/data/lambda/data/download_genomes-2025-12-18_15.00.06.699337.swarm 
 run it from bash



#### checking if everything is completely downloaded (repeat till you get no bad IDs)

In [51]:
minsize=1e9
allIDS=NULL
badIDs=NULL
for (i in 1:length(assembliesJSON$reports)){
    assemblyID=assembliesJSON$reports[[i]]$accession
    allIDS=addToVec(allIDS, assemblyID)
    dirpath=paste0(StaticInpDir, assemblyID,"/")
    fs=list.files(dirpath, pattern = "*.fna$", full.names = TRUE)
    nobigfiles=TRUE
    for (file_path in fs){
        filemeta <- file.info(file_path)
        if (filemeta$size > minsize){
            nobigfiles=FALSE
        }
    }
    if (nobigfiles){
        message("Problem with ", assemblyID)
        badIDs=addToVec(badIDs, assemblyID)
    }
}

In [52]:
length(badIDs)

[1] 0

In [50]:
swarmparams="#SWARM -t 5 -g 10 --time=10:00:00 --gres=lscratch:10"
scriptname="~/data/lambda/notebooks/download_genome.sh"
swarmlines=c(swarmparams)
for (i in 1:length(badIDs)){
    assemblyID=badIDs[i]
    tmpline=paste0(scriptname," ", assemblyID, " ", StaticInpDir, " ", sample(1:300, 1))
    swarmlines=addToVec(swarmlines, tmpline)
}

fname=paste0(StaticInpDir,"download_genomes_attempt3-", getTimeStamp(),".swarm")
writeLines(text = swarmlines, con=fname)
message("resulting file: ", fname, " \n run it from bash")

resulting file: ~/data/lambda/data/download_genomes_attempt3-2025-12-18_16.01.48.012308.swarm 
 run it from bash

